In [2]:
!pip install -q beautifulsoup4 requests

In [7]:
import os
import re
import json
import time
import requests

from bs4 import BeautifulSoup

from urllib.parse import (
    urljoin,
    urlparse,
    urlunparse
)
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0"
    )
}

PAUSA = 0.5

In [8]:
NOMBRE_PROGRAMA = "JSONs_Admision_grado.ipynb"      # cambiar si procede

ruta_programa = None

for root, dirs, files in os.walk("/content/drive/MyDrive"):

    if NOMBRE_PROGRAMA in files:
        ruta_programa = root
        break

if ruta_programa is None:
    raise Exception(
        "No se ha encontrado el notebook."
    )

CARPETA_JSON = os.path.join(
    ruta_programa,
    "JSONs"
)

os.makedirs(
    CARPETA_JSON,
    exist_ok=True
)

In [9]:
# ==========================================================
# BLOQUE 4. Páginas padre de Admisión a grado
# ==========================================================

PROFUNDIDAD_MAX = 1


FUENTES = [

    (
        "Bachillerato",
        "https://www.upv.es/admision/admision-grado/bachillerato-es.html"
    ),

    (
        "Ciclos formativos",
        "https://www.upv.es/admision/admision-grado/ciclos-formativos-es.html"
    ),

    (
        "Titulados universitarios",
        "https://www.upv.es/admision/admision-grado/titulados-universitarios-es.html"
    ),

    (
        "Mayores de 25/40/45 años",
        "https://www.upv.es/admision/admision-grado/mayores-25-40-45-es.html"
    ),

    (
        "Vengo de otra universidad",
        "https://www.upv.es/admision/admision-grado/vengo-de-otra-universidad-es.html"
    )

]

In [10]:
def limpiar_texto(txt):

    return re.sub(
        r"\s+",
        " ",
        txt
    ).strip()


def normalizar_url(url):

    p = urlparse(url)

    return urlunparse(
        (
            "https",
            p.netloc.lower().replace("www.", ""),
            p.path.rstrip("/"),
            "",
            "",
            ""
        )
    )


def get(url):

    try:

        r = requests.get(
            url,
            headers=HEADERS,
            timeout=20
        )

        r.raise_for_status()

        time.sleep(PAUSA)

        return BeautifulSoup(
            r.text,
            "html.parser"
        )

    except Exception:

        return None

In [14]:
# ==========================================================
# BLOQUE 6. Crawl de una página padre
# ==========================================================

from collections import deque


def recorrer_fuente(url_inicio):

    dominio = urlparse(url_inicio).netloc

    EXCLUIR = [

        "/legal/",
        "/otros/accesibilidad",
        "/otros/mapa-web",
        "/covid",
        "/emergencia",
        "/pictograma",
        "/comu/",
        "/china",
        "/rankings/",
        "/upv-360",
        "/nueva-imagen",
        "/aviso-legal",
        "/politica-cookies",
        "/politica-privacidad",
        "metabus",
        "busca_persona",
        "sic_dir"

    ]


    cola = deque()

    cola.append(

        (
            url_inicio,
            0,
            None,
            None
        )

    )


    visitadas = set()

    en_cola = {

        normalizar_url(
            url_inicio
        )

    }


    paginas = {}


    while cola:

        url, nivel, padre, texto_enlace = cola.popleft()

        url = normalizar_url(url)

        if url in visitadas:
            continue

        visitadas.add(url)

        print(f"[{nivel}] {url}")

        soup = get(url)
        print(soup.find("main"))
        if soup is None:
            continue


        titulo = ""

        if soup.title:

            titulo = limpiar_texto(
                soup.title.get_text()
            )


        paginas[url] = {

            "titulo": titulo,
            "url": url,
            "nivel": nivel,
            "padre": padre,
            "texto_enlace": texto_enlace,
            "enlaces": []

        }


        # ==================================================
        # SOLO en la página padre recorremos las seis secciones
        # ==================================================

        if nivel == 0:

            enlaces_html = []

            for i in range(1, 7):

                section = soup.find(

                    "section",

                    id=f"section-{i:02d}"

                )

                if section is None:
                    continue

                enlaces_html.extend(

                    section.find_all(
                        "a",
                        href=True
                    )

                )

        else:

            enlaces_html = soup.find_all(

                "a",

                href=True

            )


        # ==================================================

        for a in enlaces_html:

            href = a["href"]

            if (

                href.startswith("#")

                or href.startswith("mailto:")

                or href.startswith("javascript:")

                or href.startswith("tel:")

            ):

                continue


            destino = normalizar_url(

                urljoin(
                    url,
                    href
                )

            )


            destino_lower = destino.lower()


            if (

                destino_lower.endswith("-va.html")

                or destino_lower.endswith("-en.html")

                or "idioma=va" in destino_lower

                or "idioma=en" in destino_lower

                or "lang=va" in destino_lower

                or "lang=en" in destino_lower

            ):

                continue


            if any(

                patron in destino_lower

                for patron in EXCLUIR

            ):

                continue


            texto = limpiar_texto(

                a.get_text(
                    " ",
                    strip=True
                )

            )


            paginas[url]["enlaces"].append(

                {

                    "texto": texto,

                    "url": destino

                }

            )


            if nivel >= PROFUNDIDAD_MAX:
                continue


            p = urlparse(destino)

            if p.netloc != dominio:
                continue


            if re.search(

                r"\.(pdf|doc|docx|xls|xlsx|ppt|pptx|zip|rar|7z|jpg|jpeg|png|gif|svg|ico|mp4|avi|mov)$",

                destino,

                re.I

            ):

                continue


            if re.search(

                r"\.(css|js|xml|rss)$",

                destino,

                re.I

            ):

                continue


            if (

                destino in visitadas

                or destino in en_cola

            ):

                continue


            cola.append(

                (

                    destino,

                    nivel + 1,

                    url,

                    texto

                )

            )


            en_cola.add(destino)


    for pagina in paginas.values():

        vistos = set()

        enlaces = []

        for e in pagina["enlaces"]:

            if e["url"] in vistos:
                continue

            vistos.add(e["url"])

            enlaces.append(e)

        pagina["enlaces"] = enlaces


    return sorted(

        paginas.values(),

        key=lambda x: (

            x["nivel"],

            x["titulo"]

        )

    )

In [13]:
# ==========================================================
# BLOQUE 7. Generación del JSON
# ==========================================================

padres = []

for nombre, url in FUENTES:

    print()
    print("=" * 70)
    print(nombre)
    print("=" * 70)

    paginas = recorrer_fuente(url)

    padres.append(

        {

            "titulo": nombre,

            "url": url,

            "total": len(
                paginas
            ),

            "paginas": paginas

        }

    )


datos = {

    "fuente": "https://www.upv.es/admision/admision-grado/",

    "total": sum(

        padre["total"]

        for padre in padres

    ),

    "padres": padres

}


ruta = os.path.join(

    CARPETA_JSON,

    "admision_grado.json"

)


with open(

    ruta,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        datos,

        f,

        ensure_ascii=False,

        indent=2

    )


print()

print(
    f"Guardado en: {ruta}"
)

print()

for padre in padres:

    print(
        f"{padre['titulo']}: {padre['total']} páginas"
    )

print()

print(
    f"Total (incluyendo páginas repetidas entre padres): {datos['total']}"
)

print()

print("Proceso terminado.")


Bachillerato
[0] https://upv.es/admision/admision-grado/bachillerato-es.html

Ciclos formativos
[0] https://upv.es/admision/admision-grado/ciclos-formativos-es.html

Titulados universitarios
[0] https://upv.es/admision/admision-grado/titulados-universitarios-es.html

Mayores de 25/40/45 años
[0] https://upv.es/admision/admision-grado/mayores-25-40-45-es.html

Vengo de otra universidad
[0] https://upv.es/admision/admision-grado/vengo-de-otra-universidad-es.html

Guardado en: /content/drive/MyDrive/TFG Teleco/JSONs/admision_grado.json

Bachillerato: 0 páginas
Ciclos formativos: 0 páginas
Titulados universitarios: 0 páginas
Mayores de 25/40/45 años: 0 páginas
Vengo de otra universidad: 0 páginas

Total (incluyendo páginas repetidas entre padres): 0

Proceso terminado.
